# PillTally: train the pill-form classifier (tablet / caplet / mini tablet)

Same recipe as `tools/train_classifier.py`: YOLO11n-cls pretrained on ImageNet, fine-tuned on masked
crops of each pill. Training crops include 2 extra copies per pill with a slightly resized / shifted
box, so the classifier doesn't depend on exactly how the counter draws its box.

**Before running**, upload **`pilltally_crops_for_colab.zip`** (from the project folder) to Google Drive,
folder **`MyDrive/PillTally/`**. Then Runtime → **Run all**. The result is saved to Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/PillTally'
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!pip -q install ultralytics
import os
if not os.path.exists('/content/crops/train'):
    !unzip -q -o "/content/drive/MyDrive/PillTally/pilltally_crops_for_colab.zip" -d /content
for s in ['train', 'val', 'test']:
    print(s, {c: len(os.listdir(f'/content/crops/{s}/{c}')) for c in sorted(os.listdir(f'/content/crops/{s}'))})

In [ ]:
from ultralytics import YOLO
model = YOLO('yolo11n-cls.pt')
model.train(data='/content/crops', imgsz=224, epochs=20, batch=32, patience=10, workers=4,
            project=f'{DRIVE}/runs', name='pill_type_v3', exist_ok=True,
            degrees=180, flipud=0.5, fliplr=0.5, hsv_v=0.4, hsv_s=0.2, hsv_h=0.0)  # colour is the label: little colour jitter
import shutil
shutil.copy(f'{DRIVE}/runs/pill_type_v3/weights/best.pt', f'{DRIVE}/pill_type_v3.pt')
print('Saved', f'{DRIVE}/pill_type_v3.pt')

## Accuracy on the held-out test sessions (s03, s04, s08, s14, s19, s24)

In [ ]:
from collections import Counter
from pathlib import Path
m = YOLO(f'{DRIVE}/pill_type_v3.pt')
conf = Counter()
for folder in sorted(Path('/content/crops/test').iterdir()):
    for r in m.predict(sorted(map(str, folder.glob('*.jpg'))), imgsz=224, stream=True, verbose=False):
        conf[(folder.name, m.names[int(r.probs.top1)])] += 1
classes = sorted({t for t, _ in conf})
print(' ' * 14 + ''.join(f'{c:>14s}' for c in classes))
for t in classes:
    row = [conf[(t, p)] for p in classes]
    print(f'{t:14s}' + ''.join(f'{v:14d}' for v in row) + f'   accuracy {conf[(t, t)] / sum(row):.1%}')
print(f'overall {sum(conf[(c, c)] for c in classes) / sum(conf.values()):.1%}')
print('Download MyDrive/PillTally/pill_type_v3.pt into results/models/ on the Mac.')